# KK1 rymduppskjutningar 1957 - 2026

Datan jag använder kommer från GCAT(General Catalog of Artificial Space Objects) som är ihop satt av astrofysikern Jonathan C. Mcdowell, han har dokumenterat varenda raketuppskjutning, banändring och reentry sen 1989 i mer detalj än någon annan offentlig källa och är den forskaren som många mainstream medier går till, som exempel BBC, NYT och Reuters. Jag har primärt använt mig av satcat datasetet som listar alla satelliter uppskjutna från 1957 till 2026, en rad är ett objekt med uppskjutningsdatum, namn, omloppsbana(perigee, apogeem, inklination). Jag använder samt också mig av launch datasetet som listar information om själva raketerna som vilken launch site den har skjutits upp ifrån och som är viktigt för som jag främst använder den till är ifall rocketlaunchen var lyckad.
Viktigt att tilllägga är att satcat är bara jordbane fokuserat, alltså inte inteplanetary missions, ifall du ser en satellite som har lämnat jordens omloppsbana så kommer inte omloppsbana värdena vara användbara.
Det finns också markeringar som kan placeras på (PF, AF, IF) som betyder att det är osäkra värden på perigee, apogee eller inklination.
Jag vill undersöka hur förlitlighet hos raketer har utvecklats över tid, hur uppskjutningstakten har förändrats dom senaste 10 åren. Jag vill också se ifall det finns något samband med att vara en värdsmakt och kunna skjuta upp saker i omloppsbana.



https://planet4589.org/space/gcat/

## Här läser jag in satcat.tsv, launch.tsv och inspekterar strukturen

### 2.1 satcat

In [49]:
import pandas as pd

sat = pd.read_csv("dataset/satcat.tsv", sep="\t", skiprows=[1])
sat.info()
sat.head()

<class 'pandas.DataFrame'>
RangeIndex: 69122 entries, 0 to 69121
Data columns (total 42 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   #JCAT         69122 non-null  str   
 1   Satcat        69122 non-null  object
 2   Launch_Tag    69122 non-null  str   
 3   Piece         69122 non-null  str   
 4   Type          69122 non-null  str   
 5   Name          69122 non-null  str   
 6   PLName        69122 non-null  str   
 7   LDate         69122 non-null  str   
 8   Parent        69122 non-null  str   
 9   SDate         69122 non-null  str   
 10  Primary       69122 non-null  str   
 11  DDate         69122 non-null  str   
 12  Status        69122 non-null  str   
 13  Dest          69122 non-null  str   
 14  Owner         69122 non-null  str   
 15  State         69122 non-null  str   
 16  Manufacturer  69122 non-null  str   
 17  Bus           69122 non-null  str   
 18  Motor         69122 non-null  str   
 19  Mass          6

C:\Users\jacob\AppData\Local\Temp\ipykernel_38924\3959697528.py:3: DtypeWarning: Columns (0: Satcat, 1: Mass, 2: DryMass, 3: TotMass, 4: Length, 5: Diameter, 6: Span, 7: Perigee, 8: Inc) have mixed types. Specify dtype option on import or set low_memory=False.
  sat = pd.read_csv("dataset/satcat.tsv", sep="\t", skiprows=[1])


,#JCAT,Satcat,Launch_Tag,Piece,Type,Name,PLName,LDate,Parent,SDate,...,ODate,Perigee,PF,Apogee,AF,Inc,IF,OpOrbit,OQUAL,AltNames
0,S00001,1,1957 ALP,1957 ALP 1,R2,8K71PS No. M1-10 Stage 2,8K71A M1-10 (M1-1PS),1957 Oct 4,-,1957 Oct 4 1933,...,1957 Oct 4,214,,938,,65.1,,LLEO/I,-,-
1,S00002,2,1957 ALP,1957 ALP 2,P,1-y ISZ,PS-1,1957 Oct 4,S00001,1957 Oct 4 1933,...,1957 Oct 4,214,,938,,65.1,,LLEO/I,-,":RE,:RC"
2,S00003,3,1957 BET,1957 BET 1,P A,2-y ISZ,PS-2,1957 Nov 3,A00002,1957 Nov 3 0235,...,1957 Nov 3,211,,1659,,65.33,,LEO/I,-,":RE,:RC"
3,S00004,4,1958 ALP,1958 ALP,P A,Explorer I,Explorer 1,1958 Feb 1,A00004,1958 Feb 1 0355,...,1958 Feb 1,359,,2542,,33.18,,LEO/I,-,":UA,:UB,DEAL I:IA"
4,S00005,5,1958 BET,1958 BET 2,P,Vanguard I,Vanguard Test Satellite H,1958 Mar 17,S00016,1958 Mar 17 1224,...,1959 May 23,657,,3935,,34.25,,MEO,-,":UA,:VA"


#### Vad jag observerar från satcat

1. Alla kolumner från satcat är antingen en str eller object, inga numeriska värden. Det betyder att att mass, perigee, apogee, inc är text trots att dom innehåller nummer.

2. Datumkolumnerna (LDate, SDate, DDate, ODate) är på formatet "1959 Jan  2", alltså mänskligt läsbart men inte ISO. Så jag behöver `pd.to_datetime()` så jag kan göra den till rätt format när jag ska sortera och plotta över tid.


### 2.2 launch

In [45]:
launch = pd.read_csv("dataset/launch.tsv", sep="\t", skiprows=[1])
launch.info()
launch.head()

<class 'pandas.DataFrame'>
RangeIndex: 75831 entries, 0 to 75830
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   #Launch_Tag  75831 non-null  str    
 1   Launch_JD    75831 non-null  float64
 2   Launch_Date  75831 non-null  str    
 3   LV_Type      75831 non-null  str    
 4   Variant      75831 non-null  str    
 5   Flight_ID    75831 non-null  str    
 6   Flight       75831 non-null  str    
 7   Mission      75831 non-null  str    
 8   FlightCode   75831 non-null  str    
 9   Platform     75831 non-null  str    
 10  Launch_Site  75831 non-null  str    
 11  Launch_Pad   75831 non-null  str    
 12  Ascent_Site  75831 non-null  str    
 13  Ascent_Pad   75831 non-null  str    
 14  Apogee       75831 non-null  str    
 15  Apoflag      75831 non-null  str    
 16  Range        75831 non-null  str    
 17  RangeFlag    75831 non-null  str    
 18  Dest         75831 non-null  str    
 19  OrbPay       75

,#Launch_Tag,Launch_JD,Launch_Date,LV_Type,Variant,Flight_ID,Flight,Mission,FlightCode,Platform,...,Dest,OrbPay,Agency,LaunchCode,FailCode,Group,Category,LTCite,Cite,Notes
0,1942-A01,2430523.95,1942 Jun 13 1052,A-4,-,2,-,-,-,-,...,-,0.0,WEHR,MF,U,-,Test,TTaylor,-,-
1,1942-A02,2430587.97,1942 Aug 16 1115,A-4,-,3,-,-,-,-,...,-,0.0,WEHR,MF,U,-,Test,Stuhlinger,RocketTeam,-
2,1942-S01,2430636.12,1942 Oct 3 1458,A-4,-,4,-,-,-,-,...,-,0.0,WEHR,MS,-,-,Test,Stuhlinger,AE1961A,-
3,1942-A03,2430653.50,1942 Oct 21,A-4,-,5,-,-,-,-,...,-,0.0,WEHR,MS,-,-,Test,AE1961A,-,-
4,1942-M01,2430672.50,1942 Nov 9,A-4,-,6,-,-,-,-,...,-,0.0,WEHR,MS,-,-,Test,AE1961A,-,-


#### Vad jag observerar från launch

1. launch har 75831 entries och satcat har 69122 entries trotts att jag förvänta mig att att satcat borde innehålla mer. Det är en skillnad med 6 709 rader speglar troligen misslyckade och suborbitala uppskjutningar samt att den innehåller uppskjutningar från hela vägen bak till 1942 och innehåller missil tester vilket är intressant för min pålitlighetsfråga senare.

2. Jag ser också att det launch har LaunchCode, Failcode, Dest och Agency som är bra för det behöver jag för att kunna svara på mina frågor.

## 3. Datatvätt

I det här avsnittet rensar och normaliserar jag datan innan jag använder den i analysen. Första steget är att titta närmare på `LaunchCode` och `FailCode` i launch-tabellen, dessa två kolumner styr min senare definition av en "lyckad uppskjutning", så jag behöver veta vilka värden som faktiskt förekommer innan jag bestämmer hur jag kategoriserar dem.

In [50]:
print(launch["LaunchCode"].value_counts())
print(launch["FailCode"].value_counts())

LaunchCode
SS      39963
MS      12965
SU       8740
OS       6512
AS       2162
MU       1450
MF       1134
SF        832
AU        450
OF        425
YS        285
DS        225
TS        164
YF        136
OF40       67
AF         63
RS         60
HS         35
TF         32
OS75       19
OE         16
ME         14
TU         12
RF         11
XS         11
HF          9
!           7
OF25        5
OS80        5
OF50        4
OS85        3
OF20        2
TE          1
OS66        1
OF70        1
OS60        1
OS90        1
OF45        1
OF60        1
OF30        1
SE          1
OF44        1
OS89        1
OS65        1
OS95        1
Name: count, dtype: int64
FailCode
-       73068
U        2021
S1        125
S2P        64
S1P        61
        ...  
S2P?        1
S4?         1
S4P5        1
S4I?        1
S4P6        1
Name: count, Length: 79, dtype: int64
